In [3]:
print("Initialization")

Initialization


In [2]:
import modified_didppy as m_dp
import numpy as np
from scipy.sparse.csgraph import minimum_spanning_tree
import pulp
import itertools
import sys
import os


# 1. Simple problem

In [4]:
print(f"--- Testing DIDPPy ---")
print(f"Python executable: {sys.executable}")
print(f"DIDPPy module location: {os.path.abspath(m_dp.__file__)}")

# 1. Create a minimal model
model = m_dp.Model()
#
# --- THIS IS THE FIX ---
#
# Create an integer variable 'x'. Set the START STATE to 5.
x = model.add_int_var(target=5)
# Set the GOAL CONDITION to x == 0.
model.add_base_case([x == 0])
#
# --- END OF FIX ---
#
model.add_transition(
    m_dp.Transition(
        name="decrement",
        cost=1 + m_dp.IntExpr.state_cost(), # Cost is 1 per step
        effects=[(x, x - 1)]
    )
)
# This line is not needed for forward search, so we remove it.
# model.target_state[x] = 5 

print("Model created. Start state x=5, Target state x=0.")

# 2. Add a standard Rust expression bound
model.add_dual_bound(x) 
print("Added Rust-based dual bound (returns x)")

# 3. Define your Python function dual bound
def bound_A(state):
    return float(1)
def bound_B(state):
    return float(2)
def my_python_bound(state):
    try:
        val = state[x]
        bound = float(val * 2) # Return a float, as the evaluator expects
        # print(f"Python func called: state[x]={val}, returning bound={bound}")
        bound = bound + bound_A(state) + bound_B(state)
        return bound
    except Exception as e:
        print(f"Error in Python function: {e}")
        return None

print("Defined Python-based dual bound (returns float(x * 2))")

# 4. Instantiate your NEW solver
try:
    # This is your new class from customized_cabs_ver1.rs
    solver = m_dp.CustomDualBoundCABSv1(
        model, 
        dual_bound_func=my_python_bound, 
        quiet=False # Set to False to see the solver logs
    )
    print("\nSuccessfully created CustomDualBoundCABSv1 solver.")
except AttributeError:
    print("\n--- !! ERROR !! ---")
    print("Could not find 'dp.CustomDualBoundCABSv1'.")
    print("This means your Rust code is not installed correctly.")
    print("Please run `maturin develop` in the `didp-rs-dev/didppy` folder.")
    sys.exit(1)
except Exception as e:
    print(f"An error occurred creating the solver: {e}")
    sys.exit(1)

# 5. Run the search
print("Starting search...")
solution = solver.search()

# 6. Verify the result
print("\n--- Search Finished ---")
print(f"Transitions: {[t.name for t in solution.transitions]}")
print(f"Cost: {solution.cost}")

# The cost from x=5 to x=0 is 5 steps of cost 1.
assert solution.cost == 5
print("\n✅ Test Passed: The solution cost is correct!")

--- Testing DIDPPy ---
Python executable: c:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\venv\Scripts\python.exe
DIDPPy module location: c:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\venv\Lib\site-packages\modified_didppy\__init__.py
Model created. Start state x=5, Target state x=0.
Added Rust-based dual bound (returns x)
Defined Python-based dual bound (returns float(x * 2))

Successfully created CustomDualBoundCABSv1 solver.
Starting search...

--- Search Finished ---
Transitions: ['decrement', 'decrement', 'decrement', 'decrement', 'decrement']
Cost: 5

✅ Test Passed: The solution cost is correct!


# 2. Testing with MST bounds of CVRP model

In [ ]:
# =========================================================
# Define Python Dual Bound Function
# =========================================================

#MST dual bound for CVRP
def compute_dynamic_mst(state):
    """Compute MST cost for the current state's unvisited customers + depot."""
    try:
        # Get the list of unvisited customer IDs *from the state*
        unvisited_customers_list = list(state[unvisited_var]) 
        # Create the list of nodes for the MST (depot 0 + unvisited)
        nodes_for_mst = [0] + unvisited_customers_list
        if len(nodes_for_mst) <= 1:
            return 0.0 # No MST needed
        # Create the sub-matrix for these nodes
        sub_matrix = distance_matrix_np[np.ix_(nodes_for_mst, nodes_for_mst)]
        # Compute and return the MST cost
        mst = minimum_spanning_tree(sub_matrix)
        # Return as a float, as our evaluator expects
        return float(mst.sum())
    except Exception as e:
        print(f"Error in Python dual bound: {e}")
        return None # Return None on failure

print("Python dual bound function `compute_dynamic_mst` defined.")

#LP Relaxation dual bound for CVRP
def lp_relaxation_bound(state, cost_matrix, demand, capacity):
    """
    Compute LP relaxation lower bound for subset U (customers + depot).
    Variables x[i,j] ∈ [0,1], continuous.
    Minimize Σ_i Σ_j c[i][j]*x[i][j]
    s.t. flow conservation, capacity constraints, and subtour elimination.
    """
    U = list(state[unvisited_var]) 
    if not U:
        return 0.0
    # Include depot (node 0) in the subset for LP formulation
    Vp_set = set(U) | {0}
    Vp = sorted(list(Vp_set))
    n_sub = len(Vp)
    # Map original indices to subproblem indices
    original_to_sub = {v: i for i, v in enumerate(Vp)}
    sub_to_original = {i: v for i, v in enumerate(Vp)}
    LP_relax_model = pulp.LpProblem("CVRP_LP_Relaxation", pulp.LpMinimize)
    # --- Variables ---
    # x[i,j]: binary variable, 1 if vehicle travels from i to j, 0 otherwise (relaxed to continuous [0,1])
    x = pulp.LpVariable.dicts("x", (Vp, Vp), lowBound=0, upBound=1, cat='Continuous')
    # u[i]: continuous variable representing the "position" or "rank" of node i in a subtour (for MTZ)
    # Only needed for customer nodes in the subset U
    u = pulp.LpVariable.dicts("u", U, lowBound=0, cat='Continuous')
    # --- Objective ---
    LP_relax_model += pulp.lpSum(cost_matrix[i][j] * x[i][j] for i in Vp for j in Vp if i != j)
    # --- Constraints ---
    # 1. Each customer in U must be entered and exited exactly once
    for i in U:
        LP_relax_model += pulp.lpSum(x[i][j] for j in Vp if j != i) == 1
        LP_relax_model += pulp.lpSum(x[j][i] for j in Vp if j != i) == 1

    # 2. Depot (node 0) balance (vehicles leave and return to depot)
    # The number of vehicles leaving the depot must equal the number entering.
    # This is implicitly handled by customer constraints and flow conservation,
    # but can be explicitly stated if desired (though not strictly necessary for this relaxation).
    # model += pulp.lpSum(x[0][j] for j in U) == pulp.lpSum(x[j][0] for j in U)

    # 3. Capacity constraints (for edges leaving the depot)
    # Sum of demands on edges leaving the depot must not exceed vehicle capacity.
    # This is a weak constraint in this formulation but can be included.
    # model += pulp.lpSum(demand[j] * x[0][j] for j in U) <= capacity

    # 4. Capacity constraints (for edges entering any node - weak)
    # Sum of demands on edges entering node i <= Q
    # for i in Vp:
    #     model += pulp.lpSum(demand[j] * x[j][i] for j in Vp if j != i) <= 
        
    # 5. Subtour elimination constraints (MTZ formulation)
    # For every pair of distinct customer nodes i, j in U:
    # u[i] - u[j] + n_sub * x[i][j] <= n_sub - 1
    # where n_sub is the number of nodes in the subset (including depot)
    M_mtz = n_sub # A large enough number, can be n_sub or |U| + 1
    M_val = 10^6
    for i in U:
        for j in U:
            if i != j:
                # u_i - u_j + M * x_ij <= M - d_j
                # A common form is u_i - u_j + n * x_ij <= n - 1 for customer i,j
                # Let's use |U| for M in the constraint
                LP_relax_model = len(U)
                LP_relax_model += u[i] - u[j] + M_val * x[i][j] <= M_val - (1 if j in U else 0) # Adding 1 for consistency if depot was in u
                # Alternative using n_sub:
                # model += u[i] - u[j] + n_sub * x[i][j] <= n_sub - 1 # This is the standard MTZ for a single tour TSP
    # --- No self-loops ---
    for i in Vp:
        LP_relax_model += x[i][i] == 0
    # --- Solve ---
    # Use a solver that handles continuous variables
    solver = pulp.PULP_CBC_CMD(msg=False)
    LP_relax_model.solve(solver)
    if pulp.LpStatus[model.status] == "Optimal":
        return pulp.value(model.objective)
    else:
        # Handle cases where the LP is infeasible or unbounded
        print(f"Warning: LP for subset {U} status: {pulp.LpStatus[model.status]}")
        return float("inf") # Or some other indicator of non-optimality

Python dual bound function `compute_dynamic_mst` defined.


In [ ]:
print(f"--- Testing DIDPPy ---")
print(f"Python executable: {sys.executable}")
print(f"DIDPPy module location: {os.path.abspath(m_dp.__file__)}")

# =========================================================
# 1️⃣ Define Data (MOVED TO TOP)
# =========================================================
# Number of locations (0 is depot)
n = 4
# Number of vehicles
m = 2
# Capacity of a vehicle
q = 5
# Weights (d[0] is 0 for depot)
d = [0, 2, 3, 3]

# Distance matrix as a list of lists for the table
distance_list = [
    [0, 3, 4, 5],
    [3, 0, 5, 4],
    [4, 5, 0, 3],
    [5, 4, 3, 0]
]
# Distance matrix as a NumPy array for the Python function
distance_matrix_np = np.array(distance_list)

print("Data defined.")

# =========================================================
# 2️⃣ Define DIDP model
# =========================================================
model = m_dp.Model()

# --- Variables ---
customer = model.add_object_type(number=n)
unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name = 'unvisited_customers')
location_var = model.add_element_var(object_type=customer, target=0)
load_var = model.add_int_resource_var(target=0, less_is_better=True)
vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)

# --- Tables ---
weight = model.add_int_table(d)
distance_table = model.add_int_table(distance_list)

# --- Base case ---
# Goal is to be unvisited_var is empty AND at the depot (location 0)
model.add_base_case([unvisited_var.is_empty(), location_var == 0])

# --- Transitions ---
for j in range(1, n):
    visit = m_dp.Transition(
        name=f"visit {j}",
        cost=distance_table[location_var, j] + m_dp.IntExpr.state_cost(),
        effects=[
            (unvisited_var, unvisited_var.remove(j)),
            (location_var, j),
            (load_var, load_var + weight[j]),
        ],
        preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
    )
    model.add_transition(visit)

for j in range(1, n):
    visit_via_depot = m_dp.Transition(
        name=f"visit {j} with new vehicle",
        cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.IntExpr.state_cost(),
        effects=[
            (unvisited_var, unvisited_var.remove(j)),
            (location_var, j),
            (load_var, weight[j]), # Load resets to just this customer
            (vehicles_var, vehicles_var + 1),
        ],
        preconditions=[unvisited_var.contains(j), vehicles_var < m],
    )
    model.add_transition(visit_via_depot)

return_to_depot = m_dp.Transition(
    name="return",
    cost=distance_table[location_var, 0] + m_dp.IntExpr.state_cost(),
    effects=[(location_var, 0)],
    preconditions=[unvisited_var.is_empty(), location_var != 0],
)
model.add_transition(return_to_depot)

# --- State constraint ---
# Fixed: Need to use .sum() for a set variable index
model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])

print("DIDP Model created.")


# =========================================================
# 4️⃣ Solve model (Using your NEW Custom CABS)
# =========================================================
print("\nInstantiating CustomDualBoundCABSv1...")
try:
    # We do NOT add the bound to the model.
    # We pass the Python function *directly to the solver*
    solver = m_dp.CustomDualBoundCABSv1(
        model, 
        dual_bound_func=compute_dynamic_mst, 
        quiet=False
    )
    print("Successfully created solver. Starting search...")
    solution = solver.search()
    
except AttributeError:
    print("\n--- !! ERROR !! ---")
    print("Could not find 'dp.CustomDualBoundCABSv1'.")
    print("This means your Rust code is not installed correctly.")
    print("Please run `maturin develop` in the `didp-rs-dev/didppy` folder")
    print("and RESTART your Jupyter kernel.")
    sys.exit(1)
except Exception as e:
    print(f"An error occurred creating/running the solver: {e}")
    sys.exit(1)

# =========================================================
# 5️⃣ Print solution and transitions
# =========================================================
print("\n=== Optimal Solution ===")
if solution.cost is not None:
    print(f"Best cost: {solution.cost}")
    print("\n=== Transition sequence ===")
    for t in solution.transitions:
        print(f" - {t.name}")
else:
    print("No solution found.")

print("\n✅ Test finished.")

--- Testing DIDPPy ---
Python executable: c:\Users\ACER\Desktop\Code\0.Thesis implementation\DIDP_custom_search_guidance_local\Thesis_modified_DIDP\venv\Scripts\python.exe
DIDPPy module location: c:\Users\ACER\Desktop\Code\0.Thesis implementation\DIDP_custom_search_guidance_local\Thesis_modified_DIDP\venv\Lib\site-packages\modified_didppy\__init__.py
Data defined.
DIDP Model created.
Python dual bound function `compute_dynamic_mst` defined.

Instantiating CustomDualBoundCABSv1...
Successfully created solver. Starting search...

=== Optimal Solution ===
Best cost: 20

=== Transition sequence ===
 - visit 1
 - visit 3
 - visit 2 with new vehicle
 - return

✅ Test finished.


In [8]:
#LP Relaxation dual bound for CVRP
def lp_relaxation_bound(state, cost_matrix, demand, capacity):
    """
    Compute LP relaxation lower bound for subset U (customers + depot).
    Variables x[i,j] ∈ [0,1], continuous.
    Minimize Σ_i Σ_j c[i][j]*x[i][j]
    s.t. flow conservation, capacity constraints, and subtour elimination.
    """
    U = list(state[unvisited_var]) 
    if not U:
        return 0.0
    # Include depot (node 0) in the subset for LP formulation
    Vp_set = set(U) | {0}
    Vp = sorted(list(Vp_set))
    n_sub = len(Vp)
    # Map original indices to subproblem indices
    original_to_sub = {v: i for i, v in enumerate(Vp)}
    sub_to_original = {i: v for i, v in enumerate(Vp)}
    LP_relax_model = pulp.LpProblem("CVRP_LP_Relaxation", pulp.LpMinimize)
    # --- Variables ---
    # x[i,j]: binary variable, 1 if vehicle travels from i to j, 0 otherwise (relaxed to continuous [0,1])
    x = pulp.LpVariable.dicts("x", (Vp, Vp), lowBound=0, upBound=1, cat='Continuous')
    # u[i]: continuous variable representing the "position" or "rank" of node i in a subtour (for MTZ)
    # Only needed for customer nodes in the subset U
    u = pulp.LpVariable.dicts("u", U, lowBound=0, cat='Continuous')
    # --- Objective ---
    LP_relax_model += pulp.lpSum(cost_matrix[i][j] * x[i][j] for i in Vp for j in Vp if i != j)
    # --- Constraints ---
    # 1. Each customer in U must be entered and exited exactly once
    for i in U:
        LP_relax_model += pulp.lpSum(x[i][j] for j in Vp if j != i) == 1
        LP_relax_model += pulp.lpSum(x[j][i] for j in Vp if j != i) == 1

    # 2. Depot (node 0) balance (vehicles leave and return to depot)
    # The number of vehicles leaving the depot must equal the number entering.
    # This is implicitly handled by customer constraints and flow conservation,
    # but can be explicitly stated if desired (though not strictly necessary for this relaxation).
    # model += pulp.lpSum(x[0][j] for j in U) == pulp.lpSum(x[j][0] for j in U)

    # 3. Capacity constraints (for edges leaving the depot)
    # Sum of demands on edges leaving the depot must not exceed vehicle capacity.
    # This is a weak constraint in this formulation but can be included.
    # model += pulp.lpSum(demand[j] * x[0][j] for j in U) <= capacity

    # 4. Capacity constraints (for edges entering any node - weak)
    # Sum of demands on edges entering node i <= Q
    # for i in Vp:
    #     model += pulp.lpSum(demand[j] * x[j][i] for j in Vp if j != i) <= 
        
    # 5. Subtour elimination constraints (MTZ formulation)
    # For every pair of distinct customer nodes i, j in U:
    # u[i] - u[j] + n_sub * x[i][j] <= n_sub - 1
    # where n_sub is the number of nodes in the subset (including depot)
    M_mtz = n_sub # A large enough number, can be n_sub or |U| + 1
    M_val = 10^6
    for i in U:
        for j in U:
            if i != j:
                # u_i - u_j + M * x_ij <= M - d_j
                # A common form is u_i - u_j + n * x_ij <= n - 1 for customer i,j
                # Let's use |U| for M in the constraint
                LP_relax_model = len(U)
                LP_relax_model += u[i] - u[j] + M_val * x[i][j] <= M_val - (1 if j in U else 0) # Adding 1 for consistency if depot was in u
                # Alternative using n_sub:
                # model += u[i] - u[j] + n_sub * x[i][j] <= n_sub - 1 # This is the standard MTZ for a single tour TSP
    # --- No self-loops ---
    for i in Vp:
        LP_relax_model += x[i][i] == 0
    # --- Solve ---
    # Use a solver that handles continuous variables
    solver = pulp.PULP_CBC_CMD(msg=False)
    LP_relax_model.solve(solver)
    if pulp.LpStatus[model.status] == "Optimal":
        return pulp.value(model.objective)
    else:
        # Handle cases where the LP is infeasible or unbounded
        print(f"Warning: LP for subset {U} status: {pulp.LpStatus[model.status]}")
        return float("inf") # Or some other indicator of non-optimality

In [13]:
print(f"--- Testing DIDPPy with LP Relaxation ---")

# =========================================================
# 1️⃣ Define Data
# =========================================================
n = 4
m = 2
q = 5
# Weights (demand)
d = [0, 2, 3, 3]

# Distance matrix
distance_list = [
    [0, 3, 4, 5],
    [3, 0, 5, 4],
    [4, 5, 0, 3],
    [5, 4, 3, 0]
]
distance_matrix_np = np.array(distance_list)

print("Data defined.")

# =========================================================
# 2️⃣ Define DIDP model
# =========================================================
model = m_dp.Model()

customer = model.add_object_type(number=n)
unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name='unvisited_customers')
location_var = model.add_element_var(object_type=customer, target=0)
load_var = model.add_int_resource_var(target=0, less_is_better=True)
vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)

weight = model.add_int_table(d)
distance_table = model.add_int_table(distance_list)

model.add_base_case([unvisited_var.is_empty(), location_var == 0])

for j in range(1, n):
    visit = m_dp.Transition(
        name=f"visit {j}",
        cost=distance_table[location_var, j] + m_dp.IntExpr.state_cost(),
        effects=[
            (unvisited_var, unvisited_var.remove(j)),
            (location_var, j),
            (load_var, load_var + weight[j]),
        ],
        preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
    )
    model.add_transition(visit)

for j in range(1, n):
    visit_via_depot = m_dp.Transition(
        name=f"visit {j} with new vehicle",
        cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.IntExpr.state_cost(),
        effects=[
            (unvisited_var, unvisited_var.remove(j)),
            (location_var, j),
            (load_var, weight[j]),
            (vehicles_var, vehicles_var + 1),
        ],
        preconditions=[unvisited_var.contains(j), vehicles_var < m],
    )
    model.add_transition(visit_via_depot)

return_to_depot = m_dp.Transition(
    name="return",
    cost=distance_table[location_var, 0] + m_dp.IntExpr.state_cost(),
    effects=[(location_var, 0)],
    preconditions=[unvisited_var.is_empty(), location_var != 0],
)
model.add_transition(return_to_depot)

model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])

print("DIDP Model created.")

# =========================================================
# 3️⃣ Define LP Relaxation Dual Bound
# =========================================================

def lp_relaxation_bound(state):
    """
    Compute LP relaxation lower bound for subset U (customers + depot).
    """
    # Get unvisited customers
    U = list(state[unvisited_var])
    
    # Variables from global scope
    cost_matrix = distance_list
    
    if not U:
        # If no unvisited, cost is just returning to depot if not already there
        current_loc = state[location_var]
        if current_loc != 0:
            return float(cost_matrix[current_loc][0])
        return 0.0

    # Vp = {0} U U (Depot + unvisited customers)
    Vp_set = set(U) | {0}
    Vp = sorted(list(Vp_set))
    n_sub = len(Vp)

    # Initialize LP Problem
    # ERROR FIX: Ensure variable name is unique and not overwritten
    LP_relax_model = pulp.LpProblem("CVRP_LP_Relaxation", pulp.LpMinimize)
    
    # --- Variables ---
    # x[i][j]: Continuous [0,1]
    x = pulp.LpVariable.dicts("x", (Vp, Vp), lowBound=0, upBound=1, cat='Continuous')
    
    # u[i]: Continuous (MTZ potentials) - Only for customers
    u = pulp.LpVariable.dicts("u", U, lowBound=0, cat='Continuous')

    # --- Objective ---
    # Minimize sum of costs
    LP_relax_model += pulp.lpSum(cost_matrix[i][j] * x[i][j] for i in Vp for j in Vp if i != j)

    # --- Constraints ---
    # 1. Flow conservation: Exactly one entrance and exit for each customer
    for i in U:
        LP_relax_model += pulp.lpSum(x[i][j] for j in Vp if j != i) == 1
        LP_relax_model += pulp.lpSum(x[j][i] for j in Vp if j != i) == 1
        
    # 2. Subtour elimination (MTZ)
    M_val = n_sub # Big-M
    
    for i in U:
        for j in U:
            if i != j:
                # ERROR FIX: Removed "LP_relax_model = len(U)" which was destroying the model object
                # u_i - u_j + M * x_ij <= M - 1
                LP_relax_model += u[i] - u[j] + M_val * x[i][j] <= M_val - 1

    # 3. No self-loops
    for i in Vp:
        LP_relax_model += x[i][i] == 0

    # --- Solve ---
    # Use CBC solver, suppress output for speed
    solver = pulp.PULP_CBC_CMD(msg=False)
    LP_relax_model.solve(solver)

    # ERROR FIX: Check status of 'LP_relax_model', NOT 'model' (which is the DIDP model)
    if pulp.LpStatus[LP_relax_model.status] == "Optimal":
        return float(pulp.value(LP_relax_model.objective))
    else:
        # If infeasible or error, return 0.0 (safe lower bound)
        return 0.0

print("LP Relaxation function defined.")

# =========================================================
# 4️⃣ Solve model
# =========================================================
print("\nInstantiating CustomDualBoundCABSv1 with LP bound...")
try:
    solver = m_dp.CustomDualBoundCABSv1(
        model, 
        dual_bound_func=lp_relaxation_bound,
        quiet=False
    )
    print("Successfully created solver. Starting search...")
    solution = solver.search()
    
except Exception as e:
    import traceback
    traceback.print_exc()
    sys.exit(1)

# =========================================================
# 5️⃣ Print solution
# =========================================================
print("\n=== Optimal Solution ===")
if solution.cost is not None:
    print(f"Best cost: {solution.cost}")
    print("\n=== Transition sequence ===")
    for t in solution.transitions:
        print(f" - {t.name}")
else:
    print("No solution found.")

print("\n✅ Test finished.")

--- Testing DIDPPy with LP Relaxation ---
Data defined.
DIDP Model created.
LP Relaxation function defined.

Instantiating CustomDualBoundCABSv1 with LP bound...
Successfully created solver. Starting search...

=== Optimal Solution ===
Best cost: 20

=== Transition sequence ===
 - visit 1
 - visit 3
 - visit 2 with new vehicle
 - return

✅ Test finished.
